# Heston Model: Detailed Presentation

## Introduction

The Heston model is a widely used stochastic volatility model for pricing options. It was introduced by Steven Heston in 1993 to capture empirical features of option prices that the Black-Scholes model cannot explain, such as volatility smiles and skew.

The key idea is that the variance of the underlying asset is itself stochastic and mean-reverts to a long-term level. This allows the model to produce a richer set of implied volatility surfaces.

## Model Specification

The Heston model describes the joint evolution of the underlying asset price $S_t$ and its instantaneous variance $v_t$:

- Asset price dynamics:
    $$
    dS_t = \mu S_t\,dt + \sqrt{v_t}\,S_t\,dW_t^S
    $$

- Variance dynamics:
    $$
    dv_t = \kappa(\theta - v_t)\,dt + \sigma\sqrt{v_t}\,dW_t^v
    $$

- Correlation:
    $$
    dW_t^S \cdot dW_t^v = \rho\,dt
    $$

## Parameters

- $ \mu $: drift of the underlying asset under the real-world measure.
- $ \kappa $: speed of mean reversion of variance.
- $ \theta $: long-term mean variance (mean reversion level).
- $ \sigma $: volatility of variance, also called "volatility of volatility".
- $ \rho $: correlation between the asset returns and variance shocks.
- $ v_0 $: initial variance at time $t=0$.

In risk-neutral pricing, the drift $\mu$ is replaced by the risk-free rate $r$, and a market price of volatility risk may be incorporated.

## Key Properties

- Stochastic volatility: variance is not constant and evolves randomly.
- Mean reversion: variance tends to revert toward $\theta$.
- Leverage effect: negative correlation $\rho$ can generate the observed skew in equity option markets.
- Positivity of variance: the square-root diffusion helps keep $v_t$ non-negative, provided the Feller condition holds:
    $$
    2\kappa\theta \ge \sigma^2
    $$

## Characteristic Function and Option Pricing

The Heston model admits a semi-analytical solution for European option prices via its characteristic function. Under the risk-neutral measure, the price of a European call option can be expressed as:

$$
C(S_0, K, T) = S_0 P_1 - K e^{-rT} P_2
$$

where $P_1$ and $P_2$ are risk-neutral probabilities obtained from the characteristic function of $\ln S_T$.

The characteristic function has a closed-form expression involving complex-valued functions and solves a Riccati-type differential equation. This is one reason why the Heston model remains practical: it balances realism and tractability.

## Calibration

Calibration means choosing parameters $(\kappa, \theta, \sigma, \rho, v_0)$ to fit observed option prices or implied volatilities.

Common calibration steps:

1. Choose a parametric form or initial guesses.
2. Use market prices of liquid options across strikes and maturities.
3. Minimize an objective function, typically mean squared error between model and market implied volatilities or prices.
4. Enforce parameter constraints, such as $\theta > 0$, $\sigma > 0$, $-1 < \rho < 1$, and often the Feller condition.

Calibration can be stable if performed across a broad set of maturities and strikes.

## Applications

- Pricing European-style options.
- Modeling volatility surfaces.
- Risk management and hedging strategies.
- Valuing exotic derivatives when a stochastic volatility process is needed.
- Market making and model risk analysis.

## Advantages

- Captures volatility smile and skew.
- Provides a relatively simple analytical structure for pricing European options.
- Allows correlation between price and volatility.
- More realistic than constant volatility models.

## Limitations

- Tail behavior may still be imperfect for extreme strikes.
- Calibration can be sensitive to initial guess and market regime.
- Semi-analytical pricing is efficient for plain-vanilla options, but numerical methods are needed for many exotic products.
- The square-root variance process may not fully capture real-world variance dynamics in some markets.

## Practical Notes

- Use Fourier inversion or characteristic function methods for fast pricing.
- For path-dependent payoffs, Monte Carlo simulation or PDE methods are common.
- The model parameters have intuitive economic meanings, making it easier to interpret calibration results.

## Summary

The Heston model is a cornerstone stochastic volatility model in quantitative finance. It extends the Black-Scholes framework by letting variance follow a mean-reverting square-root process. The model is valued for its ability to produce implied volatility smiles and skew, while still allowing semi-analytical option pricing through its characteristic function.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Parametri del Modello di Heston ---
S0 = 100.0     # Prezzo iniziale dell'asset
V0 = 0.04      # Varianza iniziale (es. 20% di volatilità: 0.20^2 = 0.04)
mu = 0.05      # Tasso di rendimento atteso (drift)
kappa = 2.0    # Velocità di mean reversion della varianza
theta = 0.04   # Varianza di lungo periodo
sigma = 0.3    # Volatilità della varianza ("vol of vol")
rho = -0.7     # Correlazione (solitamente negativa per le azioni: l'effetto leva)
T = 1.0        # Orizzonte temporale in anni
N = 252        # Numero di step temporali (es. 252 giorni di trading)
M = 10         # Numero di traiettorie da simulare

dt = T / N

# --- 2. Inizializzazione degli array ---
S = np.zeros((N + 1, M))
V = np.zeros((N + 1, M))
S[0] = S0
V[0] = V0

np.random.seed(42) # Seed per rendere la simulazione riproducibile

# --- 3. Simulazione (Schema di Eulero-Maruyama) ---
for i in range(1, N + 1):
    # Generazione di due Moti Browniani indipendenti
    Z1 = np.random.standard_normal(M)
    Z2 = np.random.standard_normal(M)
    
    # Creazione dei Moti Browniani correlati (Scomposizione di Cholesky)
    W_S = Z1
    W_V = rho * Z1 + np.sqrt(1 - rho**2) * Z2
    
    # Assicuriamo che la varianza non diventi negativa nel calcolo della radice (Full Truncation)
    V_prev = np.maximum(V[i-1], 0)
    
    # Dinamica della Varianza (Processo CIR)
    V[i] = V[i-1] + kappa * (theta - V_prev) * dt + sigma * np.sqrt(V_prev) * np.sqrt(dt) * W_V
    
    # Dinamica del Prezzo (Moto Browniano Geometrico)
    S[i] = S[i-1] * np.exp((mu - 0.5 * V_prev) * dt + np.sqrt(V_prev) * np.sqrt(dt) * W_S)

# --- 4. Creazione del Grafico ---
time = np.linspace(0, T, N + 1)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Grafico dei prezzi
ax1.plot(time, S, lw=1.5)
ax1.set_title('Traiettorie del Prezzo S(t) - Modello di Heston')
ax1.set_ylabel('Prezzo')
ax1.grid(True, alpha=0.3)

# Grafico della varianza
ax2.plot(time, np.maximum(V, 0), lw=1.5)
ax2.set_title('Traiettorie della Varianza V(t) - Modello di Heston')
ax2.set_xlabel('Tempo (Anni)')
ax2.set_ylabel('Varianza')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()